# Supplementary Results 7 — gPS predicts trial safety independently of directional discordance

Does the gene pleiotropy score still predict trial-safety gene membership once directional
discordance is in the model? Variant-level discordance (1 - concordance) is aggregated over the
L2G-prioritised genes of each variant, by mean and by maximum, and three logistic regressions are
fitted on the 8,285 disease-associated genes.

Numbers are written to `results/sr07_gps_discordance.json`.

**Provenance.** `chapters/_legacy/02-analysis/05-gene-level-ps/05_safety_pleiotropy.ipynb`: the
same aggregation, the same `1 - concordance` definition with missing concordance treated as 1, and
`log2(uniqueDiseases)` as the pleiotropy term.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from manuscript_methods import paper

numbers = {}
SAFETY_SET = "trial_safety_concern"

## Gene-level discordance

A variant with no concordance information is fully concordant, so its discordance is 0 — that is
what the section's own text states, and it is the rule used for the amended column too.

**The registered numbers are the amended, sign-gated concordance** (`signedLeadDirectionalConcordance`
with missing filled to 1), adopted 2026-08-22. The published `betaSignConcordance` column and the
first redefinition are still fitted and printed beside it so both earlier lineages stay computable,
but they are no longer what S7.01-S7.09 report. `tools/expected_numbers.tsv` keeps the published
values, so the ids that moved read MISMATCH by design.

In [2]:
COLUMNS = [
    "variantId",
    "betaSignConcordance",
    "leadDirectionalConcordance",
    "leadConcordanceDefined",
    "signedLeadDirectionalConcordance",
    "prioritisedGenes",
]
variants = pd.read_parquet(paper.derived("variant_features"), columns=COLUMNS)


def aggregate(frame, column):
    """Minimum and mean concordance over the variants prioritising each gene."""
    return (
        frame.explode("prioritisedGenes")
        .dropna(subset=["prioritisedGenes"])
        .rename(columns={"prioritisedGenes": "geneId"})
        .groupby("geneId")[column]
        .agg(concordanceMin="min", concordanceMean="mean", variants="count")
        .reset_index()
    )


per_gene = aggregate(
    variants.assign(betaSignConcordance=variants["betaSignConcordance"].fillna(1.0)), "betaSignConcordance"
)
# Redefinition, dropping the variants whose concordance is undefined.
redefined_dropped = aggregate(variants[variants["leadConcordanceDefined"]], "leadDirectionalConcordance")
# Redefinition, filling undefined concordance with 1 as the published column does.
redefined_filled = aggregate(
    variants.assign(leadDirectionalConcordance=variants["leadDirectionalConcordance"].fillna(1.0)),
    "leadDirectionalConcordance",
)
# The amendment, with no concordance information treated as concordance 1.
amended_filled = aggregate(
    variants.assign(signedLeadDirectionalConcordance=variants["signedLeadDirectionalConcordance"].fillna(1.0)),
    "signedLeadDirectionalConcordance",
)
print(f"genes with a prioritised variant: {len(per_gene):,}")
print(
    f"redefinition, undefined dropped: {len(redefined_dropped):,} genes | "
    f"undefined filled with 1: {len(redefined_filled):,} genes | "
    f"amended, undefined filled with 1: {len(amended_filled):,} genes"
)

genes with a prioritised variant: 8,285
redefinition, undefined dropped: 8,049 genes | undefined filled with 1: 8,285 genes | amended, undefined filled with 1: 8,285 genes


In [3]:
genes = pd.read_parquet(paper.derived("gene_table"), columns=["geneId", "uniqueDiseases"])
gene_sets = pd.read_parquet(paper.derived("gene_sets"))
safety = set(gene_sets.loc[gene_sets["geneSet"] == SAFETY_SET, "geneId"])


def build(aggregated):
    """The model frame for one concordance aggregation."""
    frame = genes.merge(aggregated, on="geneId", how="left")
    frame["safety"] = frame["geneId"].isin(safety).astype(int)
    frame["meanDiscordance"] = 1 - frame["concordanceMean"].fillna(1.0)
    frame["maxDiscordance"] = 1 - frame["concordanceMin"].fillna(1.0)
    frame["log2gPS"] = np.log2(frame["uniqueDiseases"])
    return frame


model_frame = build(per_gene)
FRAMES = {
    "published betaSignConcordance": model_frame,
    "redefined, undefined dropped": build(redefined_dropped),
    "redefined, undefined filled with 1": build(redefined_filled),
    "amended sign-gated, undefined filled with 1": build(amended_filled),
}

# gPS keeps counting every disease term, so the gene universe and the gPS term must not move.
for label, frame in FRAMES.items():
    assert len(frame) == 8285, label
    assert frame["log2gPS"].equals(model_frame["log2gPS"]), label

print(f"genes: {len(model_frame):,} | trial-safety genes among them: {model_frame['safety'].sum():,}")
model_frame[["log2gPS", "meanDiscordance", "maxDiscordance"]].describe().round(3)

genes: 8,285 | trial-safety genes among them: 245


,log2gPS,meanDiscordance,maxDiscordance
count,8285.000,8285.000,8285.000
mean,1.392,0.009,0.046
std,1.358,0.040,0.132
min,0.000,0.000,0.000
25%,0.000,0.000,0.000
50%,1.000,0.000,0.000
75%,2.322,0.000,0.000
max,7.209,0.500,1.000


## The three models, for each discordance measure

Model 1 is gPS alone, model 2 discordance alone, model 3 the two together.

In [4]:
def logistic(frame, covariates, label):
    """Logistic regression of trial-safety membership on the given covariates."""
    design = sm.add_constant(frame[covariates])
    fitted = sm.Logit(frame["safety"], design).fit(disp=False)
    return [
        {
            "model": label,
            "term": term,
            "beta": round(float(fitted.params[term]), 4),
            "se": round(float(fitted.bse[term]), 4),
            "P": float(fitted.pvalues[term]),
        }
        for term in covariates
    ]


def fit_all(frame):
    """The five models of this section, for one concordance aggregation."""
    return pd.DataFrame(
        logistic(frame, ["log2gPS"], "gPS alone")
        + logistic(frame, ["meanDiscordance"], "mean discordance alone")
        + logistic(frame, ["maxDiscordance"], "maximum discordance alone")
        + logistic(frame, ["log2gPS", "meanDiscordance"], "gPS + mean discordance")
        + logistic(frame, ["log2gPS", "maxDiscordance"], "gPS + maximum discordance")
    )


TABLES = {label: fit_all(frame) for label, frame in FRAMES.items()}
results = TABLES["published betaSignConcordance"]
results

,model,term,beta,se,P
0,gPS alone,log2gPS,0.2907,0.0432,1.736184e-11
1,mean discordance alone,meanDiscordance,1.7556,1.2747,1.684264e-01
2,maximum discordance alone,maxDiscordance,1.8228,0.3690,7.831036e-07
3,gPS + mean discordance,log2gPS,0.2888,0.0440,5.420508e-11
4,gPS + mean discordance,meanDiscordance,0.3561,1.6115,8.250979e-01
5,gPS + maximum discordance,log2gPS,0.2526,0.0506,5.930772e-07
6,gPS + maximum discordance,maxDiscordance,0.6536,0.4442,1.411799e-01


In [5]:
# ADOPTED: the registered numbers are now the amended, sign-gated concordance, with a variant
# carrying no concordance information treated as concordance 1 -- which is what this section's own
# text states. `results` still holds the published-column fit and is printed beside it below.
REGISTERED = "amended sign-gated, undefined filled with 1"
registered = TABLES[REGISTERED]


def value(model, term, column):
    """One coefficient or P value out of the registered table."""
    row = registered[(registered["model"] == model) & (registered["term"] == term)].iloc[0]
    return float(row[column])


numbers["S7.01"] = len(FRAMES[REGISTERED])
numbers["S7.02"] = round(value("gPS + mean discordance", "log2gPS", "beta"), 2)
numbers["S7.03"] = value("gPS + mean discordance", "log2gPS", "P")
numbers["S7.04"] = round(value("gPS + mean discordance", "meanDiscordance", "P"), 2)
numbers["S7.05"] = round(value("gPS + maximum discordance", "log2gPS", "beta"), 2)
numbers["S7.06"] = value("gPS + maximum discordance", "log2gPS", "P")
numbers["S7.07"] = round(value("gPS + maximum discordance", "maxDiscordance", "P"), 2)
numbers["S7.08"] = round(value("maximum discordance alone", "maxDiscordance", "beta"), 2)
numbers["S7.09"] = value("maximum discordance alone", "maxDiscordance", "P")
for key in ["S7.03", "S7.06", "S7.09"]:
    print(f"{key}: {numbers[key]:.2e}")
print({k: numbers[k] for k in ["S7.01", "S7.02", "S7.04", "S7.05", "S7.07", "S7.08"]})

# All four definitions side by side. The amended row is the registered one; the others are kept so
# the published lineage and the first redefinition stay computable.
NINE = [
    ("S7.01", None, None, None),
    ("S7.02", "gPS + mean discordance", "log2gPS", "beta"),
    ("S7.03", "gPS + mean discordance", "log2gPS", "P"),
    ("S7.04", "gPS + mean discordance", "meanDiscordance", "P"),
    ("S7.05", "gPS + maximum discordance", "log2gPS", "beta"),
    ("S7.06", "gPS + maximum discordance", "log2gPS", "P"),
    ("S7.07", "gPS + maximum discordance", "maxDiscordance", "P"),
    ("S7.08", "maximum discordance alone", "maxDiscordance", "beta"),
    ("S7.09", "maximum discordance alone", "maxDiscordance", "P"),
]


def read(table, model, term, column):
    """One coefficient or P value out of a fitted table."""
    row = table[(table["model"] == model) & (table["term"] == term)].iloc[0]
    return float(row[column])


rows = []
for label, table in TABLES.items():
    record = {"definition": label}
    for key, model, term, column in NINE:
        if model is None:
            record[key] = len(FRAMES[label])
        elif column == "P":
            record[key] = f"{read(table, model, term, column):.2e}"
        else:
            record[key] = round(read(table, model, term, column), 2)
    rows.append(record)
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

S7.03: 4.93e-11
S7.06: 2.66e-08
S7.09: 7.12e-05
{'S7.01': 8285, 'S7.02': 0.29, 'S7.04': 0.98, 'S7.05': 0.28, 'S7.07': 0.54, 'S7.08': 1.64}
                                 definition  S7.01  S7.02    S7.03    S7.04  S7.05    S7.06    S7.07  S7.08    S7.09
              published betaSignConcordance   8285   0.29 5.42e-11 8.25e-01   0.25 5.93e-07 1.41e-01   1.82 7.83e-07
               redefined, undefined dropped   8285   0.29 4.25e-11 8.59e-01   0.27 3.34e-08 4.88e-01   1.67 4.93e-05
         redefined, undefined filled with 1   8285   0.29 4.06e-11 8.46e-01   0.27 3.34e-08 4.88e-01   1.67 4.93e-05
amended sign-gated, undefined filled with 1   8285   0.29 4.93e-11 9.78e-01   0.28 2.66e-08 5.42e-01   1.64 7.12e-05


## Write the results

In [6]:
print(paper.save_results("sr07_gps_discordance", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr07_gps_discordance.json


,computed
S7.01,8.285000e+03
S7.02,2.900000e-01
S7.03,4.933985e-11
S7.04,9.800000e-01
S7.05,2.800000e-01
S7.06,2.656294e-08
S7.07,5.400000e-01
S7.08,1.640000e+00
S7.09,7.115778e-05
